**CPSC 3810: Final Project**

**Logistic Regression on Citi Bike Demand**

*Group: Denise Pawleen Cabrera, Niki Chen, Theresa Fu*

*Details: Analyzing a Citi Bike dataset to determine in demand times for Citi Bikes around New York City.*

*Methods: Building our own Logistic Regression model and optimizing with both stochastic gradient descent and momentum stochastic gradient descent.*

In [1]:
import numpy as np
import sklearn.datasets as skdata
import sklearn.metrics as skmetrics
from sklearn.linear_model import LogisticRegression as LogisticRegressionSciKit
import matplotlib.pyplot as plt
import warnings 
import time

Matplotlib is building the font cache; this may take a moment.


Implementation of our Logistic Regression model.

In [ ]:
class Optimizer(object):
    
    def __init__(self, alpha, eta, beta, opt_type):
        
        self.__alpha = alpha
        self.__eta = eta
        self.__beta = beta
        self.__optimizer = opt_type
        self.__momentum = None
        
    def __compute_gradients(self, x, y):
        
        # using compute gradient function
        # -(1/N)((yx)/1+exp(yw^Tx))
        
        z = np.matmul(w.T, x)
        yz = y * z
        denominator = 1 + np.exp(yz)
        fraction = y / denominator
        N = x.shape[1]
        gradient = -np.matmul(x, fraction.T) / N
        
        return gradient
    
    def __polynomial_decay(self, time_step):
        
        # computes polynomial decay schedule in order to adjust learning rates
        # alpha/time_step^-eta
        
        p_decay = self.__alpha / (time_step ** -self.__eta)
        return p_decay
    
    def update(self, x, y, w, batch_size, time_step):
        
        opt = self.__optimizer
        
        if opt == 'stochastic_gradient_descent':
            
            # randomly choose indices for batch 
            batch_idx = np.random.choice(x.shape[1], size=batch_size, replace=False)
            x_batch = x[:, batch_idx]
            y_batch = y[:, batch_idx]\
            
            gradients = self.__compute_gradients(x_batch, y_batch)
            w = w - self.__polynomial_decay(time_step) * gradients
            
            return w
            
        elif opt == 'momentum_stochastic_gradient_descent':
            
            # randomly choose indices for batch
            batch_idx = np.random.choice(x.shape[1], size=batch_size, replace=False)
            x_batch = x[:, batch_idx]
            y_batch = y[:, batch_idx]
            
            gradients = self.__compute_gradients(x, y)
            
            if self.__momentum == None:
                self.__momentum = np.zeros_like(gradients)
                
            self.__momentum = self.__beta * self.__momentum + (1 - self.__beta) * gradients
            w = w - self.__polynomial_decay(time_step) * self.__momentum
            
            return w
        
        else:
            raise ValueError('Unsupported optimizer type: {}'.format(self.__optimizer))

In [ ]:
class LogisticRegression(object):
    
    def __init__(self):
        
        self.__weights = None
        self.__optimizer = None
        
    def fit(self, x, y, T, alpha, eta, beta, batch_size, opt_type, log_steps='1000', verbose='True'):
        self.__optimizer = Optimizer(alpha, eta, beta, opt_type)
        self.__weights = np.zeros((x.shape[0], 1), dtype=np.float32)
        loss_values = []

        for t in range(1, T + 1):
            self.__weights = self.__optimizer.update(self.__weights, x, y, batch_size, t)
            if (t % log_steps) == 0:
                loss = self.__compute_loss(x, y)
                loss_values.append(loss)

                if verbose:
                    print('Step={}  Loss={}'.format(t, loss))
    
        return loss_values
    
    def predict(self, x):
        z = np.dot(self.__weights.T, x)
        p = 1.0 / (1.0 + np.exp(-z))
        return (p >= 0.5).astype(np.float32)
    
    def __compute_loss(self, x, y, loss_func):
        if loss_func == 'logistic':
            z = np.dot(self.__weights.T, x)
            p = 1.0 / (1.0 + np.exp(-z))
            loss = -np.mean(y * np.log(p + 1e-8) + (1.0 - y) * np.log(1.0 - p + 1e-8))
        else:
            raise ValueError('Unsupported loss function: {}'.format(loss_func))

        return loss